
## Location Map

In [0]:
dbutils.widgets.text("data_title", "")
dbutils.widgets.text("file_store", "")

In [0]:
bronze_location = 'abfss://bronze@walmartpstorageaccount.dfs.core.windows.net/'
silver_location = 'abfss://silver@walmartpstorageaccount.dfs.core.windows.net/'

In [0]:
file_store = dbutils.widgets.get("file_store")
data_title = dbutils.widgets.get("data_title")


### Data Read

In [0]:
from pyspark.sql.functions import *

In [0]:
dfs = {}

In [0]:
cl_name = {
    'orders': 'order_id',
    'customers': 'customer_id',
    'employees': 'employee_id',
    'order_items': 'order_item_id',
    'products': 'product_id',
    'stores': 'store_id'
}

In [0]:
dfs[data_title] = spark.read.format('parquet').load(f'{bronze_location}{data_title}/{file_store}/')

In [0]:
dfs[data_title].display()


**Bronze schema**

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS walmart;


In [0]:
%sql
USE CATALOG walmart

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze;

In [0]:

spark.sql(f"""
          CREATE TABLE IF NOT EXISTS bronze.{data_title}
          USING DELTA
          AS
          SELECT
            *
          FROM parquet.`{bronze_location}{data_title}/{file_store}/`
          """
)

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:

spark.sql(f"""
          INSERT INTO bronze.{data_title}
          SELECT
            *
          FROM parquet.`{bronze_location}{data_title}/{file_store}/`
          WHERE updated_timestamp > (SELECT MAX(updated_timestamp) FROM bronze.{data_title})
          """)

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]


**Silver Schema**

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS silver;

In [0]:
spark.sql(f""" 
          CREATE TABLE IF NOT EXISTS silver.{data_title}
          USING DELTA
          LOCATION '{silver_location}{data_title}/{file_store}/'
          AS
          SELECT
              * EXCEPT (rank)
          FROM(SELECT 
                  *,
                  ROW_NUMBER() OVER(PARTITION BY {cl_name[data_title]} ORDER BY updated_timestamp DESC) AS `rank`,
                  current_timestamp() AS `processed_timestamp`
              FROM bronze.{data_title})t
          WHERE rank = 1
          """)


DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql(f"""
          WITH CTE_1 AS (
            SELECT
                * EXCEPT (rank)
            FROM (
                SELECT
                    *,
                    ROW_NUMBER() OVER (PARTITION BY {cl_name[data_title]} ORDER BY updated_timestamp DESC) AS rank,
                    current_timestamp() AS processed_timestamp
                FROM bronze.{data_title}
            ) t
            WHERE rank= 1 AND updated_timestamp > (SELECT MAX(updated_timestamp) FROM silver.{data_title})
                )
            
            MERGE INTO silver.{data_title} AS t
            USING CTE_1 AS s
            ON t.{cl_name[data_title]} = s.{cl_name[data_title]}
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
          """)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]